In [7]:
import pandas as pd

#读取员工数据
df = pd.read_csv(r"/Users/blucechen/PycharmProjects/pythontest/data/employees.csv")
df

#按department_id 分组。计算salary 的最小值，中位数，最大值
df.groupby("department_id")["salary"].agg(["min","median","max"])

,min,median,max
department_id,,,
10.0,4400.0,4400.0,4400.0
20.0,6000.0,9500.0,13000.0
30.0,2500.0,2850.0,11000.0
40.0,6500.0,6500.0,6500.0
50.0,2100.0,3100.0,8200.0
60.0,4200.0,4800.0,9000.0
70.0,10000.0,10000.0,10000.0
80.0,6100.0,8900.0,14000.0
90.0,17000.0,17000.0,24000.0


In [15]:
#针对不同的聚合字段进行不同的聚合操作
# 按department_id 分组，统计job_id 的种类数，commission_pct的平均值
df.groupby("department_id").agg({"job_id":"nunique","commission_pct":"mean"})

# df.groupby("department_id")[["job_id","commission_pct"]].agg(["nunique","max"])

df.groupby("department_id").agg({"job_id":"nunique","commission_pct":"mean"}).rename(columns={"job_id":"岗位数","commission_pct":"平均值"})

df.groupby("department_id").agg(岗位数=("job_id","nunique"),平均值=("commission_pct","mean"))


,岗位数,平均值
department_id,,
10.0,1,NaN
20.0,2,NaN
30.0,2,NaN
40.0,1,NaN
50.0,3,NaN
60.0,1,NaN
70.0,1,NaN
80.0,2,0.225
90.0,2,NaN


In [17]:
#自定义聚合函数操作
#统计每个部门员工lastname 首字母
def f(x):
    result = set()
    for i in x:
        result.add(i[0])
    return result

df.groupby("department_id")["last_name"].agg(f)

department_id
10.0                                                   {W}
20.0                                                {F, H}
30.0                                    {B, T, C, K, R, H}
40.0                                                   {M}
50.0     {C, N, E, W, B, K, A, J, L, V, M, S, T, D, O, ...
60.0                                       {E, A, P, H, L}
70.0                                                   {B}
80.0     {C, E, Z, B, K, A, J, L, M, V, S, T, D, P, O, ...
90.0                                                {D, K}
100.0                                   {S, U, C, F, P, G}
110.0                                               {G, H}
Name: last_name, dtype: object

### transform分组转换

In [20]:
import pandas as pd

#读取员工数据
df = pd.read_csv(r"/Users/blucechen/PycharmProjects/pythontest/data/employees.csv")
df

df.groupby("department_id")["salary"].transform(lambda x:x - x.mean())

0      4666.666667
1     -2333.333333
2     -2333.333333
3      3240.000000
4       240.000000
          ...     
102   -3500.000000
103       0.000000
104       0.000000
105    1850.000000
106   -1850.000000
Name: salary, Length: 107, dtype: float64

In [31]:
#2）通过transform()按分组使用平均值填充缺失值
import pandas as pd
import numpy as np
df = pd.read_csv(r"/Users/blucechen/PycharmProjects/pythontest/data/employees.csv")
df

na_index = pd.Series(df.index.tolist()).sample(30)
na_index

df.loc[na_index, "salary"] = pd.NA
print(df.groupby("department_id")["salary"].agg(["size","count"])) #查看每组数据总数与非空数据数

def fill_missing(x):
    #使用平均值填充
    if np.isnan(x.mean()):
        return x.fillna(0)
    return x.fillna(x.mean())
df["salary"] = df.groupby("department_id")["salary"].transform(fill_missing)
print(df.groupby("department_id")["salary"].agg(["size","count"]))

               size  count
department_id             
10.0              1      0
20.0              2      1
30.0              6      4
40.0              1      0
50.0             45     32
60.0              5      4
70.0              1      1
80.0             34     27
90.0              3      1
100.0             6      4
110.0             2      2
               size  count
department_id             
10.0              1      1
20.0              2      2
30.0              6      6
40.0              1      1
50.0             45     45
60.0              5      5
70.0              1      1
80.0             34     34
90.0              3      3
100.0             6      6
110.0             2      2


In [33]:
#分组过滤

commission_pct_filter = df.groupby("department_id").filter(
  lambda x: x["commission_pct"].notnull().all()
) # 按department_id分组，过滤掉commission_pct包含空值的分组
print(commission_pct_filter)

    employee_id   first_name   last_name     email        phone_number  \
45          145         John     Russell   JRUSSEL  011.44.1344.429268   
46          146        Karen    Partners  KPARTNER  011.44.1344.467268   
47          147      Alberto   Errazuriz  AERRAZUR  011.44.1344.429278   
48          148       Gerald   Cambrault  GCAMBRAU  011.44.1344.619268   
49          149        Eleni     Zlotkey  EZLOTKEY  011.44.1344.429018   
50          150        Peter      Tucker   PTUCKER  011.44.1344.129268   
51          151        David   Bernstein  DBERNSTE  011.44.1344.345268   
52          152        Peter        Hall     PHALL  011.44.1344.478968   
53          153  Christopher       Olsen    COLSEN  011.44.1344.498718   
54          154      Nanette   Cambrault  NCAMBRAU  011.44.1344.987668   
55          155       Oliver     Tuvault  OTUVAULT  011.44.1344.486508   
56          156      Janette        King     JKING  011.44.1345.429268   
57          157      Patrick       Sul